# 12 — Optimized Strategy Report

**Perubahan dari v10:**
- SL: 1.0x ATR → **1.2x ATR**
- Trend detection lebih ketat: slope > 0.00025 + EMA50 > EMA200 + close > EMA200

**Target:** Win Rate >= 50%, Profit Factor >= 1.4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

CSV = '../../app/ai/candle_ai/EURUSD.m_M15_202112160545_202603251830.csv'
df_raw = pd.read_csv(CSV, sep='\t')
df_raw.columns = [c.strip('<>').lower() for c in df_raw.columns]
df_raw['datetime'] = pd.to_datetime(df_raw['date'] + ' ' + df_raw['time'])
df_raw = df_raw.set_index('datetime').sort_index()
df_raw = df_raw[['open','high','low','close','tickvol']].rename(columns={'tickvol':'vol'})
print(f'M15: {len(df_raw):,} candles | {df_raw.index[0]} -> {df_raw.index[-1]}')

In [ ]:
# ── Indikator ──────────────────────────────────────────────────────────────
def ema(s, p): return s.ewm(span=p, adjust=False).mean()
def atr(df, p=14):
    h,l,c = df['high'],df['low'],df['close']
    tr = pd.concat([h-l,(h-c.shift()).abs(),(l-c.shift()).abs()],axis=1).max(axis=1)
    return tr.ewm(span=p,adjust=False).mean()
def macd_hist(s,fast=12,slow=26,sig=9):
    ml = s.ewm(span=fast,adjust=False).mean()-s.ewm(span=slow,adjust=False).mean()
    return ml-ml.ewm(span=sig,adjust=False).mean()

# H1
df_h1 = df_raw.resample('1h').agg({'open':'first','high':'max','low':'min','close':'last'}).dropna()
df_h1['ema50']  = ema(df_h1['close'],50)
df_h1['ema200'] = ema(df_h1['close'],200)
df_h1['atr_h1'] = atr(df_h1,14)
df_h1['slope']  = df_h1['ema50'].diff(3)

# Trend KETAT: slope threshold + EMA50 vs EMA200 + close vs EMA200
SLOPE_THR = 0.00025
df_h1['trend'] = 'sideways'
df_h1.loc[(df_h1['slope']>SLOPE_THR)&(df_h1['close']>df_h1['ema200'])&(df_h1['ema50']>df_h1['ema200']),'trend'] = 'up'
df_h1.loc[(df_h1['slope']<-SLOPE_THR)&(df_h1['close']<df_h1['ema200'])&(df_h1['ema50']<df_h1['ema200']),'trend'] = 'down'

print('Trend distribution:')
print(df_h1['trend'].value_counts())

# M15
df = df_raw.copy()
df['atr']    = atr(df,14)
df['macd_h'] = macd_hist(df['close'])
df['ema9']   = ema(df['close'],9)
df['ema21']  = ema(df['close'],21)
df['trend_h1']  = df_h1['trend'].reindex(df.index,method='ffill')
df['ema50_h1']  = df_h1['ema50'].reindex(df.index,method='ffill')
df['ema200_h1'] = df_h1['ema200'].reindex(df.index,method='ffill')
df['atr_h1']    = df_h1['atr_h1'].reindex(df.index,method='ffill')
print('Indikator selesai')

In [ ]:
# ── S/R Zone H1 ────────────────────────────────────────────────────────────
SW_WIN=4; H1_LB=500; CLUST_DIST=0.0010; PROX_ATR=1.0
h1_hi=df_h1['high'].values; h1_lo=df_h1['low'].values
h1_idx=df_h1.index; nh1=len(df_h1)

is_sh=np.zeros(nh1,dtype=bool); is_sl=np.zeros(nh1,dtype=bool)
for i in range(SW_WIN,nh1-SW_WIN):
    if h1_hi[i]==max(h1_hi[i-SW_WIN:i+SW_WIN+1]): is_sh[i]=True
    if h1_lo[i]==min(h1_lo[i-SW_WIN:i+SW_WIN+1]): is_sl[i]=True

def cluster_zones(prices):
    if not prices: return []
    sp=sorted(prices); zones,cl=[],[sp[0]]
    for p in sp[1:]:
        if p-cl[-1]<=CLUST_DIST: cl.append(p)
        else: zones.append(np.mean(cl)); cl=[p]
    zones.append(np.mean(cl)); return zones

nm15=len(df); m15_cls=df['close'].values; m15_ath1=df['atr_h1'].values; m15_idx=df.index
h1_idarr=np.array(h1_idx)
in_sup=np.zeros(nm15,dtype=bool); in_res=np.zeros(nm15,dtype=bool)

prev_h1p=-1; cur_sup,cur_res=[],[]
for i in range(nm15):
    h1p=int(np.searchsorted(h1_idarr,m15_idx[i].floor('1h')))
    if h1p>=nh1: h1p=nh1-1
    if h1p!=prev_h1p and h1p>=H1_LB:
        s=h1p-H1_LB
        cur_res=cluster_zones(h1_hi[s:h1p][is_sh[s:h1p]].tolist())
        cur_sup=cluster_zones(h1_lo[s:h1p][is_sl[s:h1p]].tolist())
        prev_h1p=h1p
    c=m15_cls[i]; thr=PROX_ATR*(m15_ath1[i] if not np.isnan(m15_ath1[i]) else 0.0012)
    for z in cur_sup:
        if abs(c-z)<=thr: in_sup[i]=True; break
    for z in cur_res:
        if abs(c-z)<=thr: in_res[i]=True; break

df['in_sup']=in_sup; df['in_res']=in_res
print('S/R Zone OK')

In [ ]:
# ── Candle Pattern ─────────────────────────────────────────────────────────
o,h,l,c=df['open'].values,df['high'].values,df['low'].values,df['close'].values
n=len(df)
body=np.abs(c-o); up_sh=h-np.maximum(o,c); lo_sh=np.minimum(o,c)-l
pin_bull=(lo_sh>2.0*body)&(lo_sh>up_sh)&(c>(h+l)/2)
pin_bear=(up_sh>2.0*body)&(up_sh>lo_sh)&(c<(h+l)/2)
eng_bull=np.zeros(n,dtype=bool); eng_bear=np.zeros(n,dtype=bool)
for i in range(1,n):
    if c[i]>o[i] and c[i-1]<o[i-1] and c[i]>o[i-1] and o[i]<c[i-1]: eng_bull[i]=True
    if c[i]<o[i] and c[i-1]>o[i-1] and c[i]<o[i-1] and o[i]>c[i-1]: eng_bear[i]=True
df['pat_bull']=pin_bull|eng_bull; df['pat_bear']=pin_bear|eng_bear
print('Candle Pattern OK')

In [ ]:
# ── Signal Generation ──────────────────────────────────────────────────────
BEST_HOURS={2,3,8,9,10,12,13,16,17}; ATR_MIN=0.0008
trend_arr=df['trend_h1'].values; macd_arr=df['macd_h'].values
ema9_arr=df['ema9'].values; ema21_arr=df['ema21'].values
atr_arr=df['atr'].values; ins_arr=df['in_sup'].values
inr_arr=df['in_res'].values; pb_arr=df['pat_bull'].values; pp_arr=df['pat_bear'].values

macd_up=np.zeros(n,dtype=bool); macd_down=np.zeros(n,dtype=bool)
for i in range(1,n):
    if macd_arr[i]>macd_arr[i-1] and macd_arr[i]>0: macd_up[i]=True
    elif macd_arr[i]<macd_arr[i-1] and macd_arr[i]<0: macd_down[i]=True

sig_arr=np.full(n,'hold',dtype=object); score_arr=np.zeros(n,dtype=int)
hour_arr=np.array([t.hour for t in df.index])
START=H1_LB*4
for i in range(START,n):
    if atr_arr[i]<ATR_MIN: continue
    if hour_arr[i] not in BEST_HOURS: continue
    if trend_arr[i]=='up' and ins_arr[i]:
        sc=2
        if macd_up[i]: sc+=1
        if ema9_arr[i]>ema21_arr[i]: sc+=1
        if pb_arr[i]: sc+=1
        if sc>=3: sig_arr[i]='buy'; score_arr[i]=sc
    elif trend_arr[i]=='down' and inr_arr[i]:
        sc=2
        if macd_down[i]: sc+=1
        if ema9_arr[i]<ema21_arr[i]: sc+=1
        if pp_arr[i]: sc+=1
        if sc>=3: sig_arr[i]='sell'; score_arr[i]=sc

df['signal']=sig_arr; df['score']=score_arr
print(pd.Series(sig_arr).value_counts())

In [ ]:
# ── Backtest Engine (SL 1.2x ATR) ─────────────────────────────────────────
LOT=0.02; DAILY_TP=5.0; DAILY_SL=-5.0; MODAL=100.0; RR=1.5; SL_MULT=1.2; MAX_CANDLE=16
pip_val=LOT*10

cls=df['close'].values; hi=df['high'].values; lo2=df['low'].values
at=df['atr'].values; sig=df['signal'].values; sc=df['score'].values; idx=df.index

trades=[]; equity=MODAL; daily_pnl={}; daily_stop={}
i=0
while i<n:
    if sig[i] not in ('buy','sell') or sc[i]<3:
        i+=1; continue
    trade_date=idx[i].date()
    d_pnl=daily_pnl.get(trade_date,0.0)
    if daily_stop.get(trade_date,False): i+=1; continue
    if d_pnl>=DAILY_TP or d_pnl<=DAILY_SL:
        daily_stop[trade_date]=True; i+=1; continue
    d=sig[i]; entry=cls[i]
    sl_d=SL_MULT*at[i]; tp_d=RR*at[i]
    sl=entry-sl_d if d=='buy' else entry+sl_d
    tp=entry+tp_d if d=='buy' else entry-tp_d
    outcome='timeout'; ep=entry
    for j in range(i+1,min(i+1+MAX_CANDLE,n)):
        if d=='buy':
            if lo2[j]<=sl: outcome='loss'; ep=sl; break
            if hi[j]>=tp:  outcome='win';  ep=tp; break
        else:
            if hi[j]>=sl:  outcome='loss'; ep=sl; break
            if lo2[j]<=tp: outcome='win';  ep=tp; break
    pips=(ep-entry)*10000 if d=='buy' else (entry-ep)*10000
    usd=pips*pip_val; equity+=usd
    daily_pnl[trade_date]=daily_pnl.get(trade_date,0.0)+usd

    # Alasan loss (untuk analisis)
    loss_reason = ''
    if outcome == 'loss':
        sl_pips = abs(pips)
        if sl_pips <= 10: loss_reason = 'SL_ketat'
        elif sl_pips <= 15: loss_reason = 'SL_normal'
        else: loss_reason = 'SL_besar'

    trades.append({
        'No':             len(trades)+1,
        'Tanggal':        idx[i].strftime('%Y-%m-%d'),
        'Jam (UTC)':      idx[i].strftime('%H:%M'),
        'Hari':           idx[i].strftime('%A'),
        'Arah':           d.upper(),
        'Score':          int(sc[i]),
        'Entry':          round(entry,5),
        'SL':             round(sl,5),
        'TP':             round(tp,5),
        'Exit':           round(ep,5),
        'Hasil':          outcome.upper(),
        'Pips':           round(pips,1),
        'Profit/Loss ($)':round(usd,2),
        'Modal ($)':      round(equity,2),
        'PnL Hari ($)':   round(daily_pnl[trade_date],2),
        'Alasan Loss':    loss_reason,
    })
    i+=MAX_CANDLE+1

df_trades=pd.DataFrame(trades)
print(f'Total trade: {len(df_trades)}')
print(df_trades['Hasil'].value_counts())

In [ ]:
# ── Perbandingan v10 vs v12 ────────────────────────────────────────────────
total=len(df_trades)
wins=(df_trades['Hasil']=='WIN').sum()
loss=(df_trades['Hasil']=='LOSS').sum()
to=(df_trades['Hasil']=='TIMEOUT').sum()
wr=round(wins/total*100,1)
gp=df_trades.loc[df_trades['Pips']>0,'Pips'].sum()
gl=abs(df_trades.loc[df_trades['Pips']<0,'Pips'].sum() or 1)
pf=round(gp/gl,3)
total_usd=round(df_trades['Profit/Loss ($)'].sum(),2)
max_dd=round((df_trades['Modal ($)'].cummax()-df_trades['Modal ($)']).max(),2)

print('='*55)
print('  PERBANDINGAN HASIL BACKTEST')
print('='*55)
print(f'  {"Metrik":<25} {"v10 (lama)":>12} {"v12 (baru)":>12}')
print('-'*55)
print(f'  {"Total Trades":<25} {1152:>12} {total:>12}')
print(f'  {"Win Rate":<25} {"43.4%":>12} {str(wr)+"%":>12}')
print(f'  {"Profit Factor":<25} {"1.30":>12} {str(pf):>12}')
print(f'  {"Total USD":<25} {"$+374":>12} {"$"+str(total_usd):>12}')
print(f'  {"Max Drawdown":<25} {"$31.82":>12} {"$"+str(max_dd):>12}')
print(f'  {"SL":<25} {"1.0x ATR":>12} {"1.2x ATR":>12}')
print(f'  {"Slope Threshold":<25} {"none":>12} {"0.00025":>12}')
print('='*55)

In [ ]:
# ── Analisis Loss Detail ───────────────────────────────────────────────────
df_loss = df_trades[df_trades['Hasil']=='LOSS']

print('=== LOSS PER JAM UTC ===')
loss_hr = df_trades.groupby(df_trades['Jam (UTC)'].str[:2].astype(int)).apply(lambda x: pd.Series({
    'total': len(x),
    'loss':  (x['Hasil']=='LOSS').sum(),
    'wr':    round((x['Hasil']=='WIN').mean()*100,1),
    'usd':   round(x['Profit/Loss ($)'].sum(),2)
}), include_groups=False)
print(loss_hr.sort_values('wr', ascending=False).to_string())
print()

print('=== LOSS PER ARAH ===')
print(df_trades.groupby(['Arah','Hasil']).size().unstack(fill_value=0))
buy_wr = round((df_trades[df_trades['Arah']=='BUY']['Hasil']=='WIN').mean()*100,1)
sell_wr = round((df_trades[df_trades['Arah']=='SELL']['Hasil']=='WIN').mean()*100,1)
print(f'Buy WR: {buy_wr}% | Sell WR: {sell_wr}%')
print()

print('=== ALASAN LOSS ===')
print(df_loss['Alasan Loss'].value_counts())
print()

print('=== LOSS PER SCORE ===')
print(df_trades.groupby(['Score','Hasil']).size().unstack(fill_value=0))

In [ ]:
# ── Daily & Monthly Summary ────────────────────────────────────────────────
daily_rows=[]
for d, pnl in daily_pnl.items():
    dt=df_trades[df_trades['Tanggal']==str(d)]
    if len(dt)==0: continue
    daily_rows.append({
        'Tanggal':         str(d),
        'Hari':            pd.Timestamp(d).strftime('%A'),
        'Jumlah Trade':    len(dt),
        'Win':             (dt['Hasil']=='WIN').sum(),
        'Loss':            (dt['Hasil']=='LOSS').sum(),
        'Win Rate (%)':    round((dt['Hasil']=='WIN').mean()*100,1),
        'Total Pips':      round(dt['Pips'].sum(),1),
        'P&L Hari ($)':   round(pnl,2),
        'Modal Akhir ($)': round(dt['Modal ($)'].iloc[-1],2),
        'Status':          'HIT TP' if pnl>=DAILY_TP else ('HIT SL' if pnl<=DAILY_SL else 'Normal'),
    })
df_daily=pd.DataFrame(daily_rows).sort_values('Tanggal')

df_trades['Bulan']=pd.to_datetime(df_trades['Tanggal']).dt.to_period('M').astype(str)
monthly=df_trades.groupby('Bulan').apply(lambda x: pd.Series({
    'Trades':        len(x),
    'Win':           (x['Hasil']=='WIN').sum(),
    'Loss':          (x['Hasil']=='LOSS').sum(),
    'Win Rate (%)':  round((x['Hasil']=='WIN').mean()*100,1),
    'Total Pips':    round(x['Pips'].sum(),1),
    'P&L Bulan ($)': round(x['Profit/Loss ($)'].sum(),2),
    'Modal Akhir ($)':round(x['Modal ($)'].iloc[-1],2),
}), include_groups=False).reset_index()

print('=== MONTHLY ===')
print(monthly.to_string(index=False))

In [ ]:
# ── Visualisasi ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18,10))
fig.suptitle('v12 Optimized: SL 1.2x ATR | Slope Threshold 0.00025', fontsize=13, fontweight='bold')

# 1. Equity curve
ax=axes[0,0]
ax.plot(range(len(df_trades)), df_trades['Modal ($)'], color='royalblue', lw=1.5)
ax.axhline(100, color='gray', ls='--', alpha=0.5, label='Modal')
ax.fill_between(range(len(df_trades)), df_trades['Modal ($)'], 100,
                where=df_trades['Modal ($)']>=100, alpha=0.15, color='green')
ax.fill_between(range(len(df_trades)), df_trades['Modal ($)'], 100,
                where=df_trades['Modal ($)']<100, alpha=0.15, color='red')
ax.set_title('Equity Curve ($)'); ax.set_ylabel('USD'); ax.grid(True,alpha=0.3); ax.legend()

# 2. Daily P&L
ax=axes[0,1]
dpnl=pd.Series(daily_pnl); dpnl_nz=dpnl[dpnl!=0]
colors=['#2ecc71' if v>0 else '#e74c3c' for v in dpnl_nz]
ax.bar(range(len(dpnl_nz)), dpnl_nz.values, color=colors, alpha=0.8)
ax.axhline(0,color='black',lw=0.8)
ax.axhline(5,color='green',ls='--',alpha=0.5,label='TP $5')
ax.axhline(-5,color='red',ls='--',alpha=0.5,label='SL -$5')
ax.set_title('Daily P&L ($)'); ax.legend(); ax.grid(True,alpha=0.3)

# 3. WR per jam
ax=axes[0,2]
hr_wr = df_trades.copy()
hr_wr['hour'] = pd.to_datetime(hr_wr['Tanggal']+' '+hr_wr['Jam (UTC)']).dt.hour
hr = hr_wr.groupby('hour').apply(lambda x: pd.Series({
    'wr': (x['Hasil']=='WIN').mean()*100, 'n': len(x)
}), include_groups=False).reset_index()
hr = hr[hr['n']>=3]
colors_hr=['#2ecc71' if w>=50 else '#e74c3c' for w in hr['wr']]
ax.bar(hr['hour'], hr['wr'], color=colors_hr, alpha=0.8)
ax.axhline(50,color='gray',ls='--',alpha=0.5)
ax.set_title('Win Rate per Jam UTC'); ax.grid(True,alpha=0.3)

# 4. Pips per trade
ax=axes[1,0]
colors_t=['#2ecc71' if p>0 else '#e74c3c' for p in df_trades['Pips']]
ax.bar(range(len(df_trades)), df_trades['Pips'], color=colors_t, alpha=0.8)
ax.axhline(0,color='black',lw=0.8)
ax.set_title('Pips per Trade'); ax.grid(True,alpha=0.3)

# 5. Monthly P&L
ax=axes[1,1]
colors_m=['#2ecc71' if v>0 else '#e74c3c' for v in monthly['P&L Bulan ($)']]
ax.bar(range(len(monthly)), monthly['P&L Bulan ($)'], color=colors_m, alpha=0.8)
ax.axhline(0,color='black',lw=0.8)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Bulan'], rotation=90, fontsize=6)
ax.set_title('P&L per Bulan ($)'); ax.grid(True,alpha=0.3)

# 6. Buy vs Sell WR
ax=axes[1,2]
for d in ['BUY','SELL']:
    t=df_trades[df_trades['Arah']==d]
    if len(t)==0: continue
    wr_val=(t['Hasil']=='WIN').mean()*100
    ax.bar(d, wr_val, color='#3498db' if d=='BUY' else '#e74c3c', alpha=0.8)
    ax.text(d, wr_val+1, f'{wr_val:.1f}%\nn={len(t)}', ha='center', fontsize=10)
ax.axhline(50,color='gray',ls='--',alpha=0.5)
ax.set_title('WR: BUY vs SELL'); ax.set_ylim(0,80); ax.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('12_optimized_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart tersimpan: 12_optimized_result.png')

In [ ]:
# ── Export Excel ───────────────────────────────────────────────────────────
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

OUTPUT = '12_optimized_report.xlsx'
wb = openpyxl.Workbook()

GREEN=PatternFill('solid',fgColor='C6EFCE'); RED=PatternFill('solid',fgColor='FFC7CE')
YELLOW=PatternFill('solid',fgColor='FFEB9C'); BLUE=PatternFill('solid',fgColor='1F4E79')
GRAY=PatternFill('solid',fgColor='D9D9D9')
WF=Font(color='FFFFFF',bold=True); BF=Font(bold=True)
thin=Border(left=Side(style='thin'),right=Side(style='thin'),top=Side(style='thin'),bottom=Side(style='thin'))

def style_header(ws, cols):
    for col in range(1,cols+1):
        cell=ws.cell(row=1,column=col)
        cell.fill=BLUE; cell.font=WF
        cell.alignment=Alignment(horizontal='center'); cell.border=thin

def auto_width(ws):
    for col in ws.columns:
        mx=max((len(str(c.value or '')) for c in col),default=10)
        ws.column_dimensions[col[0].column_letter].width=min(mx+4,40)

# ── Sheet 1: Per Trade Log ──
ws1=wb.active; ws1.title='Per Trade Log'
for r in dataframe_to_rows(df_trades, index=False, header=True): ws1.append(r)
style_header(ws1, len(df_trades.columns))
for row in ws1.iter_rows(min_row=2, max_row=ws1.max_row):
    hasil=row[10].value
    for cell in row: cell.border=thin; cell.alignment=Alignment(horizontal='center')
    if hasil=='WIN':
        for cell in row: cell.fill=GREEN
    elif hasil=='LOSS':
        for cell in row: cell.fill=RED
    elif hasil=='TIMEOUT':
        for cell in row: cell.fill=YELLOW
auto_width(ws1); ws1.freeze_panes='A2'

# ── Sheet 2: Analisis Loss ──
ws2=wb.create_sheet('Analisis Loss')
loss_detail = df_trades[df_trades['Hasil']=='LOSS'][['No','Tanggal','Jam (UTC)','Hari','Arah','Score','Entry','SL','Exit','Pips','Profit/Loss ($)','Modal ($)','Alasan Loss']].copy()
for r in dataframe_to_rows(loss_detail, index=False, header=True): ws2.append(r)
style_header(ws2, len(loss_detail.columns))
for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
    alasan=row[12].value
    for cell in row: cell.border=thin; cell.alignment=Alignment(horizontal='center')
    if alasan=='SL_ketat': 
        for cell in row: cell.fill=PatternFill('solid',fgColor='FFE0E0')
    elif alasan=='SL_besar':
        for cell in row: cell.fill=RED
auto_width(ws2); ws2.freeze_panes='A2'

# ── Sheet 3: Daily Summary ──
ws3=wb.create_sheet('Daily Summary')
for r in dataframe_to_rows(df_daily, index=False, header=True): ws3.append(r)
style_header(ws3, len(df_daily.columns))
for row in ws3.iter_rows(min_row=2, max_row=ws3.max_row):
    status=row[9].value; pnl_val=row[7].value
    for cell in row: cell.border=thin; cell.alignment=Alignment(horizontal='center')
    if status=='HIT TP':
        for cell in row: cell.fill=GREEN
    elif status=='HIT SL':
        for cell in row: cell.fill=RED
    elif isinstance(pnl_val,(int,float)) and pnl_val>0:
        for cell in row: cell.fill=PatternFill('solid',fgColor='EBF5EB')
    elif isinstance(pnl_val,(int,float)) and pnl_val<0:
        for cell in row: cell.fill=PatternFill('solid',fgColor='FFF0F0')
auto_width(ws3); ws3.freeze_panes='A2'

# ── Sheet 4: Monthly Summary ──
ws4=wb.create_sheet('Monthly Summary')
for r in dataframe_to_rows(monthly, index=False, header=True): ws4.append(r)
style_header(ws4, len(monthly.columns))
for row in ws4.iter_rows(min_row=2, max_row=ws4.max_row):
    pnl_val=row[5].value
    for cell in row: cell.border=thin; cell.alignment=Alignment(horizontal='center')
    if isinstance(pnl_val,(int,float)) and pnl_val>0:
        for cell in row: cell.fill=GREEN
    elif isinstance(pnl_val,(int,float)) and pnl_val<0:
        for cell in row: cell.fill=RED
auto_width(ws4); ws4.freeze_panes='A2'

# ── Sheet 5: Overall Summary ──
ws5=wb.create_sheet('Overall Summary')
days_tp=(df_daily['Status']=='HIT TP').sum()
days_sl=(df_daily['Status']=='HIT SL').sum()
avg_daily=round(df_daily['P&L Hari ($)'].mean(),2)
best=df_daily.loc[df_daily['P&L Hari ($)'].idxmax()]
worst=df_daily.loc[df_daily['P&L Hari ($)'].idxmin()]

summary=[
    ('VERSI STRATEGI',    'v12 Optimized'),
    ('Perubahan',         'SL 1.2x ATR + Slope Threshold 0.00025'),
    ('',''),
    ('PARAMETER',''),
    ('Modal Awal',        '$100'),
    ('Lot',              f'{LOT} (1 pip = ${pip_val})'),
    ('SL',               f'1.2x ATR'),
    ('TP',               f'1.5x ATR (RR 1:1.25)'),
    ('Daily TP',         f'+${DAILY_TP}'),
    ('Daily SL',         f'-${abs(DAILY_SL)}'),
    ('Slope Threshold',  '0.00025'),
    ('',''),
    ('PERFORMA TRADE',''),
    ('Total Trade',       total),
    ('Win',               wins),
    ('Loss',              loss),
    ('Timeout',           to),
    ('Win Rate',         f'{wr}%'),
    ('Profit Factor',    f'{pf}'),
    ('Total Profit/Loss',f'${total_usd:+}'),
    ('Modal Akhir',      f'${round(100+total_usd,2)}'),
    ('Max Drawdown',     f'${max_dd}'),
    ('',''),
    ('PERFORMA HARIAN',''),
    ('Total Hari Trading', len(df_daily)),
    ('Hari Profit',       (df_daily['P&L Hari ($)']>0).sum()),
    ('Hari Loss',         (df_daily['P&L Hari ($)']<0).sum()),
    ('Hari Hit TP ($5+)', days_tp),
    ('Hari Hit SL (-$5)', days_sl),
    ('Avg P&L/Hari',     f'${avg_daily}'),
    ('Hari Terbaik',     f"{best['Tanggal']} (+${best['P&L Hari ($)']})"),
    ('Hari Terburuk',    f"{worst['Tanggal']} (${worst['P&L Hari ($)']})"),
    ('',''),
    ('ANALISIS LOSS',''),
    ('Loss SL ketat (<10pip)',  (df_loss['Alasan Loss']=='SL_ketat').sum()),
    ('Loss SL normal (10-15pip)',(df_loss['Alasan Loss']=='SL_normal').sum()),
    ('Loss SL besar (>15pip)',  (df_loss['Alasan Loss']=='SL_besar').sum()),
    ('Buy Win Rate',     f'{buy_wr}%'),
    ('Sell Win Rate',    f'{sell_wr}%'),
]
df_sum=pd.DataFrame(summary,columns=['Metrik','Nilai'])
for r in dataframe_to_rows(df_sum,index=False,header=True): ws5.append(r)
style_header(ws5,2)
for row in ws5.iter_rows(min_row=2,max_row=ws5.max_row):
    metrik=row[0].value
    for cell in row: cell.border=thin; cell.alignment=Alignment(horizontal='left')
    if metrik in ('VERSI STRATEGI','PARAMETER','PERFORMA TRADE','PERFORMA HARIAN','ANALISIS LOSS'):
        for cell in row: cell.fill=GRAY; cell.font=BF
ws5.column_dimensions['A'].width=30; ws5.column_dimensions['B'].width=38

wb.save(OUTPUT)
print(f'\n✅ Report tersimpan: {OUTPUT}')
print('5 Sheet:')
print('  1. Per Trade Log   — semua trade (hijau=win, merah=loss)')
print('  2. Analisis Loss   — detail setiap loss + alasannya')
print('  3. Daily Summary   — ringkasan per hari')
print('  4. Monthly Summary — ringkasan per bulan')
print('  5. Overall Summary — statistik + perbandingan v10 vs v12')